# ALQAC 2026 
**Runtime ▸ GPU (T4)**. Điểm = `0.70·Outcome + 0.20·PenalizedCaseRecall + 0.10·MicroLawF1`.

**Config đang dùng:**
- Outcome: **prompt STRICT** → 52% (bản BASE 50%).
- Case evidence: query nhắm **đoạn QUYẾT ĐỊNH/nhận định** (max_calls=8) → 24.2%. *(Thêm query tình tiết KHÔNG giúp — đã thử, tụt còn 15%.)*
- Law: `{law_id, aid}` từ BM25 → 16.3%.
- Kỳ vọng khi gộp đúng: **~0.43**.

Upload: `alqac2026_starter.zip`, `corpus_law_pub.json`, `ALQAC2026_public_test.json`.


## 1. Cài thư viện

In [ ]:
!pip -q install rank_bm25 pyvi sentence-transformers \
    "transformers>=4.44" accelerate bitsandbytes scikit-learn python-dotenv requests
print("done")

## 2. Upload & giải nén

In [ ]:
from google.colab import files
import zipfile, os
up = files.upload()
zips=[f for f in up if f.endswith('.zip')]; assert zips, "thiếu alqac2026_starter.zip"
with zipfile.ZipFile(zips[0]) as z: z.extractall('.')
os.makedirs('alqac2026/data/raw', exist_ok=True)
for f in up:
    if f.endswith('.json'): os.replace(f, f'alqac2026/data/raw/{f}')
os.chdir('alqac2026'); print("CWD:", os.getcwd(), "| data:", os.listdir('data/raw'))

## 3. Nạp dữ liệu + BM25

In [ ]:
import sys; sys.path.insert(0,'src')
os.environ['CORPUS_PATH']='data/raw/corpus_law_pub.json'
os.environ['PUBLIC_TEST_PATH']='data/raw/ALQAC2026_public_test.json'
from collections import Counter
from data_utils import load_corpus, load_public_test, LABELS
from retrieval import BM25Retriever
arts=load_corpus(os.environ['CORPUS_PATH']); cases=load_public_test(os.environ['PUBLIC_TEST_PATH'])
id2text={a.doc_id:a.text for a in arts}
print(f"Articles={len(arts)} Cases={len(cases)} Labels={dict(Counter(c.verdict_label for c in cases))}")
bm=BM25Retriever(arts); print("BM25 sẵn sàng.")

## 4. Nạp LLM 4-bit (Qwen2.5-7B-Instruct)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
MODEL="Qwen/Qwen2.5-7B-Instruct"
bnb=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                       bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
tok=AutoTokenizer.from_pretrained(MODEL)
model=AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=bnb,
                                           device_map="auto", trust_remote_code=True).eval()
print("Loaded", MODEL)

## 5. Hàm dự đoán (prompt STRICT)

In [ ]:
import json
from collections import Counter
from agent import build_user_prompt, build_few_shots, parse_label
from agent import SYSTEM_PROMPT as SYS_BASE
few = build_few_shots(cases, n_per_class=1)
SYS_STRICT = SYS_BASE + '''

PHÂN BIỆT KỸ (hay bị nhầm):
- Tòa chấp nhận TẤT CẢ yêu cầu chính của nguyên đơn -> A_WIN.
- Chấp nhận một số, bác một số -> PARTIAL_A_WIN.
- Bác TẤT CẢ yêu cầu của nguyên đơn -> B_WIN.
- Nguyên đơn chỉ được phần rất nhỏ -> PARTIAL_B_WIN.
Đọc kỹ phần Tòa 'chấp nhận / không chấp nhận / đình chỉ' trong bằng chứng.'''

def law_ctx_for(q, k=5): return [(d, id2text[d]) for d,_ in bm.search(q, k=k)]

@torch.inference_mode()
def predict(query, ctx, system_prompt=SYS_STRICT, sc=0):
    user=build_user_prompt(query, ctx, few)
    msgs=[{"role":"system","content":system_prompt},{"role":"user","content":user}]
    prompt=tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inp=tok(prompt, return_tensors="pt", truncation=True, max_length=7000).to(model.device)
    if sc and sc>1:
        votes=[]
        for _ in range(sc):
            o=model.generate(**inp, max_new_tokens=400, do_sample=True, temperature=0.7,
                             top_p=0.9, pad_token_id=tok.eos_token_id)
            votes.append(parse_label(tok.decode(o[0][inp['input_ids'].shape[1]:], skip_special_tokens=True)))
        return Counter(votes).most_common(1)[0][0]
    o=model.generate(**inp, max_new_tokens=400, do_sample=False, pad_token_id=tok.eos_token_id)
    return parse_label(tok.decode(o[0][inp['input_ids'].shape[1]:], skip_special_tokens=True))

## 6. Evidence 

In [ ]:
import re, time, requests
from retrieve_client import EvidenceRetriever
# vá retrieve() chịu lỗi 5xx
def _retrieve(self, query, case_id, max_retries=3):
    H={"X-API-Key":self.token,"Content-Type":"application/json","ngrok-skip-browser-warning":"true"}
    for attempt in range(max_retries):
        self._throttle()
        r=requests.post(f"{self.base}/retrieve", headers=H, json={"query":query,"case_id":case_id}, timeout=self.timeout)
        self._last_call=time.time(); self.n_calls+=1
        if r.status_code==200: return r.json().get("results",[])
        if r.status_code in (429,500,502,503,504): time.sleep(self.min_interval*(attempt+1)); continue
        if r.status_code==403: raise PermissionError("403 token sai")
        if r.status_code==422: raise ValueError(f"422 {r.text[:120]}")
        r.raise_for_status()
    print(f"[bỏ qua] {case_id}"); return []
EvidenceRetriever.retrieve=_retrieve

DECISION_QUERIES=[            # QUYẾT ĐỊNH/nhận định lên đầu
    "quyết định của tòa án tuyên xử",
    "chấp nhận yêu cầu khởi kiện của nguyên đơn",
    "không chấp nhận yêu cầu của nguyên đơn",
    "nhận định của hội đồng xét xử",
    "áp dụng điều luật bộ luật dân sự",
    "nghĩa vụ chịu án phí dân sự sơ thẩm",
]
def evidence_queries(case_query, max_q=8):
    qs=list(DECISION_QUERIES)                      # decision trước
    m=re.search(r'tranh chấp[^.,;]*', case_query, re.I)
    if m: qs.append(m.group(0).strip())
    qs.append(re.sub(r'\s+',' ',case_query)[:200]) # query gốc sau cùng
    seen,out=set(),[]
    for q in qs:
        k=q.lower()
        if k and k not in seen: seen.add(k); out.append(q)
        if len(out)>=max_q: break
    return out
def gather_evidence(rc, case_id, case_query, max_calls=8):
    ev={}
    for q in evidence_queries(case_query, max_q=max_calls):
        for hit in rc.retrieve(q, case_id):
            cid=hit["chunk_id"]
            if cid not in ev or hit["score"]>ev[cid]["score"]: ev[cid]={"score":hit["score"],"text":hit["text"]}
    return ev
print("evidence queries:", evidence_queries("Tranh chấp đất")[:3], "...")

## 7. Token

In [ ]:
from google.colab import userdata
try: os.environ["ALQAC_TEAM_TOKEN"]=userdata.get("ALQAC_TEAM_TOKEN")
except Exception: os.environ["ALQAC_TEAM_TOKEN"]="alqac_iFTRPVd0izrUmnQ4FbcO-_O0uxOCjBuN" 
rc_test=EvidenceRetriever()
_=gather_evidence(rc_test,"case_4101","tranh chấp bồi thường",max_calls=2)
print("token+API OK" if rc_test.n_calls else "kiểm tra token")

## 8. Law từ BM25 dạng {law_id, aid}

In [ ]:
docid2meta={a.doc_id:{"law_id":a.law_id,"aid":int(a.aid)} for a in arts}
def law_evidence_objs(query,k=5): return [docid2meta[d] for d,_ in bm.search(query,k=k)]
print("ví dụ law_evidence:", law_evidence_objs("bồi thường thiệt hại")[:2])

## 9. VÒNG TẠO SUBMISSION (một run sạch — chạy để nộp)
STRICT + evidence-quyết-định + law{law_id,aid}. ~50–60 phút.

In [ ]:
from tqdm.auto import tqdm
MAX_CALLS=8; BEST_SYS=SYS_STRICT; BEST_SC=0
official=[{"case_id":c.case_id,"case_query":c.case_query} for c in cases]
rc2=EvidenceRetriever(); submission=[]; evidence_cache={}
for c in tqdm(official):
    before=rc2.n_calls
    ev=gather_evidence(rc2, c["case_id"], c["case_query"], max_calls=MAX_CALLS)
    api_calls=rc2.n_calls-before
    ev_ctx=[(cid,h["text"]) for cid,h in ev.items()]; evidence_cache[c["case_id"]]=ev_ctx
    pred=predict(c["case_query"], ev_ctx+law_ctx_for(c["case_query"]), BEST_SYS, BEST_SC)
    submission.append({"case_id":c["case_id"],"prediction":pred,
                       "law_evidence":law_evidence_objs(c["case_query"]),
                       "case_evidence":list(ev.keys()),"api_calls":api_calls})
json.dump(evidence_cache, open("evidence_cache.json","w",encoding="utf-8"), ensure_ascii=False)
json.dump(submission, open("submission.json","w",encoding="utf-8"), ensure_ascii=False, indent=2)
gold={c.case_id:c.verdict_label for c in cases}
yt=[gold[s["case_id"]] for s in submission]; yp=[s["prediction"] for s in submission]
print(f"API calls={rc2.n_calls}  avg chunks/case={sum(len(s['case_evidence']) for s in submission)/50:.1f}")
print("Outcome Acc offline =", round(sum(a==b for a,b in zip(yt,yp))/50,3))

## 10. Chẩn đoán nhanh + tải về
Xem chunks/case & api_calls TRƯỚC khi nộp (tránh nộp bản recall thấp).

In [ ]:
print("avg chunks/case:", round(sum(len(s["case_evidence"]) for s in submission)/50,2))
print("api_calls:", Counter(s["api_calls"] for s in submission))
print("số chunk/case:", Counter(len(s["case_evidence"]) for s in submission))
# nộp nếu avg chunks >= ~4 và outcome ~0.52
from google.colab import files
files.download("submission.json")